In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("EmployeeRDD") \
    .master("local[2]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

sc = spark.sparkContext

print("Spark:", spark.version)

Spark: 3.5.8


In [2]:
employees_rdd = sc.textFile("employees_records.csv")

header = employees_rdd.first()
print(header)

data_rdd = employees_rdd.filter(lambda row: row != header)

data_rdd = data_rdd.map(lambda row: row.split(","))

print(data_rdd.take(2))

emp_id,name,department,city,age,salary,experience,gender,joining_year
[['101', 'Aarav', 'IT', 'Pune', '44', '48000', '4', 'M', '2022'], ['102', 'Diya', 'HR', 'Chennai', '32', '73000', '4', 'F', '2022']]


In [ ]:
high_salary = data_rdd.filter(lambda x: int(x[5]) > 70000)

print(high_salary.collect())

[['102', 'Diya', 'HR', 'Chennai', '32', '73000', '4', 'F', '2022'], ['103', 'Rohan', 'Finance', 'Bangalore', '27', '114000', '6', 'M', '2020'], ['105', 'Vikram', 'IT', 'Hyderabad', '26', '74000', '2', 'M', '2024'], ['106', 'Meera', 'HR', 'Chennai', '43', '116000', '1', 'F', '2025'], ['107', 'Arjun', 'Finance', 'Chennai', '44', '98000', '18', 'M', '2008'], ['109', 'Karthik', 'IT', 'Chennai', '37', '80000', '11', 'M', '2015'], ['111', 'Rahul', 'Finance', 'Mumbai', '36', '90000', '2', 'M', '2024'], ['113', 'Aditya', 'IT', 'Bangalore', '38', '93000', '4', 'M', '2022'], ['114', 'Nisha', 'HR', 'Hyderabad', '41', '91000', '10', 'F', '2016'], ['117', 'Manoj', 'IT', 'Delhi', '27', '80000', '4', 'M', '2022'], ['119', 'Surya', 'Finance', 'Pune', '35', '79000', '4', 'M', '2022'], ['121', 'Abhishek', 'IT', 'Delhi', '31', '104000', '3', 'M', '2023'], ['122', 'Neha', 'HR', 'Chennai', '32', '116000', '11', 'F', '2015'], ['124', 'Shreya', 'Sales', 'Mumbai', '25', '96000', '3', 'F', '2023'], ['125', 'Na

In [4]:
df = spark.read.csv("employees_records.csv", header=True, inferSchema=True)

df.createOrReplaceTempView("employees")
spark.sql("SELECT * FROM employees WHERE salary > 70000").show()

+------+---------+----------+---------+---+------+----------+------+------------+
|emp_id|     name|department|     city|age|salary|experience|gender|joining_year|
+------+---------+----------+---------+---+------+----------+------+------------+
|   102|     Diya|        HR|  Chennai| 32| 73000|         4|     F|        2022|
|   103|    Rohan|   Finance|Bangalore| 27|114000|         6|     M|        2020|
|   105|   Vikram|        IT|Hyderabad| 26| 74000|         2|     M|        2024|
|   106|    Meera|        HR|  Chennai| 43|116000|         1|     F|        2025|
|   107|    Arjun|   Finance|  Chennai| 44| 98000|        18|     M|        2008|
|   109|  Karthik|        IT|  Chennai| 37| 80000|        11|     M|        2015|
|   111|    Rahul|   Finance|   Mumbai| 36| 90000|         2|     M|        2024|
|   113|   Aditya|        IT|Bangalore| 38| 93000|         4|     M|        2022|
|   114|    Nisha|        HR|Hyderabad| 41| 91000|        10|     F|        2016|
|   117|    Mano

In [6]:
df = spark.read.csv("employees_records.csv", header=True, inferSchema=True)

df.filter(df.department == "IT").show()

df.createOrReplaceTempView("employees")
spark.sql("SELECT * FROM employees WHERE department = 'IT'").show()

+------+--------+----------+---------+---+------+----------+------+------------+
|emp_id|    name|department|     city|age|salary|experience|gender|joining_year|
+------+--------+----------+---------+---+------+----------+------+------------+
|   101|   Aarav|        IT|     Pune| 44| 48000|         4|     M|        2022|
|   105|  Vikram|        IT|Hyderabad| 26| 74000|         2|     M|        2024|
|   109| Karthik|        IT|  Chennai| 37| 80000|        11|     M|        2015|
|   113|  Aditya|        IT|Bangalore| 38| 93000|         4|     M|        2022|
|   117|   Manoj|        IT|    Delhi| 27| 80000|         4|     M|        2022|
|   121|Abhishek|        IT|    Delhi| 31|104000|         3|     M|        2023|
|   125|  Naveen|        IT|     Pune| 26|117000|         2|     M|        2024|
|   129|  Pranav|        IT|Hyderabad| 32| 99000|        10|     M|        2016|
|   133|   Akash|        IT|    Delhi| 37| 94000|         3|     M|        2023|
|   137|   Vivek|        IT|

In [7]:
df.createOrReplaceTempView("employees")

spark.sql("""SELECT SUM(salary) AS total_salary FROM employees""").show()

+------------+
|total_salary|
+------------+
|     4013000|
+------------+



In [9]:
spark.sql("""SELECT SUM(salary) AS total_salary, AVG(salary) AS average_salary FROM employees""").show()

+------------+--------------+
|total_salary|average_salary|
+------------+--------------+
|     4013000|       80260.0|
+------------+--------------+



In [10]:
spark.sql("""SELECT department, COUNT(*) AS employee_count FROM employees GROUP BY department ORDER BY employee_count DESC""").show()

+----------+--------------+
|department|employee_count|
+----------+--------------+
|        HR|            13|
|        IT|            13|
|     Sales|            12|
|   Finance|            12|
+----------+--------------+

